In [ ]:
import pandas as pd

from superstore_sales_dataset.config import RAW_DATA_DIR

In [ ]:
sales_data = pd.read_csv(
    RAW_DATA_DIR / "superstore_sales_dataset.csv",
    encoding="ISO-8859-1",
)

In [ ]:
sales_data.info()

# Auditoria Temporal

In [ ]:
sales_data.describe(include="all")

In [ ]:
# Verificando valores < 0 na coluna 'Profit'
negative_profit = sales_data[sales_data["Profit"] < 0]
print(f"Total de registros com lucro negativo: {len(negative_profit)}")

In [ ]:
# Verificando valores =0 na coluna 'Profit'
zero_profit = sales_data[sales_data["Profit"] == 0]
print(f"Total de registros com lucro zero: {len(zero_profit)}")

In [ ]:
# Criando uma nova coluna para indicar se a linha é de prejuízo (Profit < 0)
sales_data = sales_data.assign(negative_profit_flag=sales_data["Profit"] < 0)
sales_data["negative_profit_flag"].value_counts()

## Profit Margin Ratio

Razão matemática entre o lucro e a venda de cada registo ($\frac{\text{Profit}}{\text{Sales}}$), garantindo o tratamento de eventuais divisões por zero.

In [ ]:
sales_data["profit_margin_ratio"] = sales_data.apply(
    lambda row: row["Profit"] / row["Sales"] if row["Sales"] != 0 else 0, axis=1
)

sales_data["profit_margin_ratio"].describe()

## Is Loss Making (Indicador de Operação Deficitária):
Variável booleana (True/False) que identifica se o item do pedido gerou prejuízo ($\text{Profit} < 0$)

In [ ]:
# Criando uma nova coluna para indicar se a linha é de prejuízo (Profit < 0)
sales_data["Is Loss Making"] = sales_data.apply(
    lambda row: "True" if row["Profit"] < 0 else "False", axis=1
)

sales_data["Is Loss Making"].value_counts()

# KPIs Essenciais de Negócio

---

## Faturamento Total (Receita Bruta):
Soma global da coluna Sales ($\sum \text{Sales}$).

In [ ]:
total_revenue = sales_data["Sales"].sum()

print(f"Total Revenue: ${total_revenue:,.2f}")

## Lucro Consolidado (Lucro Operacional Líquido): 
Soma global da coluna Profit ($\sum \text{Profit}$).   

In [ ]:
consolidated_profit = sales_data["Profit"].sum()
print(f"Consolidated Profit: ${consolidated_profit:,.2f}")

## Margem Líquida Consolidada (%): 
Razão ponderada entre o Lucro Consolidado e o Faturamento Total ($\frac{\sum \text{Profit}}{\sum \text{Sales}} \times 100$), evitando o erro comum de calcular a média simples das margens individuais.

In [ ]:
consolidate_net_margin = consolidated_profit / total_revenue
print(f"Consolidated Net Margin: {consolidate_net_margin:.2%}")

## Ticket Médio: 
Valor médio de faturamento por pedido ($\frac{\sum \text{Sales}}{\text{Total de Pedidos Únicos}}$ ou por transação).

In [ ]:
mean_ticket = total_revenue / len(sales_data["Order ID"].unique())
print(f"Mean Ticket: ${mean_ticket:,.2f}")

---
## Métricas de Auditoria de Sangramento Financeiro



    ## Impacto Financeiro de Pedidos Deficitários: 
Soma do prejuízo gerado exclusivamente pelas vendas onde $\text{Profit} < 0$.

In [ ]:
# Receita proveniente de transações deficitárias
loss_order_impact = sales_data[sales_data["Is Loss Making"] == "True"]["Sales"].sum()

print(f"Total Loss Orders Impact: ${loss_order_impact:,.2f}")

## Percentual de Faturamento Drenado por Operações Deficitárias: 
Proporção da receita total que provém de transações que dão prejuízo ($\frac{\sum \text{Sales}_{\text{Profit} < 0}}{\sum \text{Sales}_{\text{Total}}} \times 100$).

In [ ]:
# Percentual de faturamento associado às operações deficitárias
loss_orders_percentage = (
    loss_order_impact / total_revenue * 100 if total_revenue != 0 else None
)

if loss_orders_percentage is not None:
    print(f"Percentage of Loss Orders: {loss_orders_percentage:.2f}%")
else:
    print("Percentage of Loss Orders: Undefined (total revenue is zero)")

## Total de Transações Deficitárias: 
Contagem absoluta e percentual de registos com margem negativa ($\text{Is Loss Making} == \text{True}$). 

In [ ]:
total_loss_sales = (
    sales_data[sales_data["Is Loss Making"] == "True"].value_counts().sum()
)
total_sales = len(sales_data)

percetage_loos_sales = total_loss_sales / total_sales * 100

print(f"Total Sales: {total_sales}")
print(f"Total Loss Sales: {total_loss_sales}")
print(f"Percentage Loss Sales: {percetage_loos_sales:.2f}%")

## Correlação Desconto vs. Margem Negativa: 
Cruzamento empírico que mede a margem líquida por categoria nas faixas de desconto superiores a 20% ($\text{Discount} > 0.20$). 

# Ranking de Subcategorias Críticas 
([RF-3] — Função obter_piores_subcategorias)

## Prejuízo / Lucro Acumulado por Subcategoria: 
Soma de Profit agrupada por Sub-Category, ordenada das maiores perdas para as menores.

## Faturamento e Margem da Subcategoria: 
Soma de Sales e margem ponderada de cada uma dessas 10 subcategorias críticas para contextualizar o volume de vendas afetado.   

# Métricas do Simulador de Sensibilidade 
([RF-4] e Secção 6 — Função simular_teto_desconto)
Indicadores recalculados dinamicamente quando o gestor define um "Desconto Máximo Permitido" ($d_{\text{teto}}$) no slider

## Faturamento Simulado: 
Nova receita total após limitar os descontos superiores ao teto estipulado.

## Lucro Simulado: 
Novo lucro operacional líquido consolidado com a política de teto de desconto aplicada.

## Margem Líquida Simulada (%): 
Nova margem ponderada após a simulação.

## Recuperação Financeira (Ganho de Lucro): 
    Diferença monetária entre o Lucro Simulado e o Lucro Consolidado original ($\text{Lucro}_{\text{Simulado}} - \text{Lucro}_{\text{Original}}$).

## Taxa de Redução de Transações com Margem Negativa (%): 
Percentual de pedidos deficitários que deixam de dar prejuízo após o teto de desconto (o KPI de sucesso do projeto exige encontrar o teto que atinja 80% de redução).

# Métrica Extra de Destaque 
(Secção 24 — Desafios de Extensão)

## Tempo de Ciclo Logístico (Lead Time): 
Diferença em dias entre a data de envio e a data do pedido ($\text{Ship Date} - \text{Order Date}$) para avaliar se atrasos na entrega têm relação com descontos compensatórios. 